# README
This notebook defines several option pricing and numerical methods functions.

- `mynormcdf(x)`: approximate the standard normal cumulative distribution function (CDF) using a polynomial approximation.
- `option_BS_c(S0, K, r, sigma, T)`: compute the Black-Scholes price of a European call option.
- `CRR_Euro_Call(S0, K, r, sigma, T, N, isRound=False, dp=4)`: price a European call option with the Cox-Ross-Rubinstein binomial tree.
- `CRR_Euro_Put(S0, K, r, sigma, T, N, isRound=False, dp=4)`: price a European put option with the CRR binomial tree.
- `CRR_Amer_Call(S0, K, r, sigma, T, N, isRound=False, dp=4)`: price an American call option with the CRR binomial tree and early exercise.
- `CRR_Amer_Put(S0, K, r, sigma, T, N, isRound=False, dp=4)`: price an American put option with the CRR binomial tree and early exercise.
- `TriTree_Euro_Call(S0, K, r, sigma, T, N, isRound=False, dp=4)`: price a European call option using a trinomial tree.
- `FDExplicit_Euro_Call(S0, K, r, sigma, T, M, N, isRound=False, dp=4)`: solve a European call option via explicit finite differences.
- `FDImplicit_Euro_Call(S0, K, r, sigma, T, M, N, isRound=False, dp=4)`: solve a European call option via implicit finite differences.

Each section below includes example calls demonstrating the computed option price.

In [ ]:
from scipy.stats import norm
from math import sqrt, exp, pi

def mynormcdf(x):
   if x>=0:
      gamma=0.2316419
      #k=1/(1+gamma*x)  #if k is not rounded
      k=round(1/(1+gamma*x), 4)
      print('k:', k)
      a1=0.319381530
      a2=-0.356563782
      a3=1.781477937
      a4=-1.821255978
      a5=1.330274429
      res=1-1/(sqrt(2*pi))*exp(-x**2/2)*(a1*k+
             a2*k**2+a3*k**3+a4*k**4+a5*k**5)
      print(f'N({x}): {res}')
      return res
   else:
      print(f'N({x})=1-N({-x})')
      return 1-mynormcdf(-x)
   
print(norm.cdf(1), round(mynormcdf(1),4))
print(norm.cdf(-1), round(mynormcdf(-1),4))


## BLACK SCHOLES OPTIONS PRICING 

In [ ]:

from scipy.stats import norm
from math import sqrt, log, exp

def option_BS_c(S0, K, r, sigma, T):
   d1=(log(S0/K)+(r+sigma**2/2)*T)/(sigma*sqrt(T))
   d2=(log(S0/K)+(r-sigma**2/2)*T)/(sigma*sqrt(T))
   c=S0*norm.cdf(d1)-K*exp(-r*T)*norm.cdf(d2)
   return c

S0, K, r, sigma, T=42, 40, 0.1, 0.2, 0.5
print(option_BS_c(S0, K, r, sigma, T))

## CRR EUROPEAN CALL OPTION 

In [ ]:
from math import exp, sqrt
def calculate(x, isRound=False, dp=4):
    return round(x,dp) if isRound else x
def CRR_Euro_Call(S0, K, r, sigma, T, N, isRound=False, dp=4):
   dt=calculate(T/N, isRound, dp)
   u=calculate(exp(sigma*sqrt(dt)), isRound, dp)
   d=calculate(1/u, isRound, dp)
   p=calculate((exp(r*dt)-d)/(u-d), isRound, dp)
   f=[[0 for j in range(0,i+1)] for i in range(0,N+1)]
   for j in range(0,N+1):
      f[N][j]=calculate(max(S0*u**j*d**(N-j)-K,0), isRound, dp)
      #print(f'f[{N}][{j}]: {f[N][j]}')
   for i in range(N-1, 0-1, -1):
      for j in range(0, i+1):
         f[i][j]=calculate(exp(-r*dt)*(p*f[i+1][j+1]+(1-p)*f[i+1][j]), isRound, dp)
         #print(f'f[{i}][{j}]: {f[i][j]}')
   return f[0][0]

S0,K,r,sigma,T,N=50, 50, 0.1, 0.4, 1, 3
print(CRR_Euro_Call(S0,K,r,sigma,T,N))
print(CRR_Euro_Call(S0,K,r,sigma,T,N,isRound=True))
print(CRR_Euro_Call(S0,K,r,sigma,T,N=12))

## CRR EUROPEAN PUT OPTION 

In [ ]:
from math import exp, sqrt
def calculate(x, isRound=False, dp=4):
    return round(x,dp) if isRound else x
def CRR_Euro_Put(S0, K, r, sigma, T, N, isRound=False, dp=4):
   dt=calculate(T/N, isRound, dp)
   u=calculate(exp(sigma*sqrt(dt)), isRound, dp)
   d=calculate(1/u, isRound, dp)
   p=calculate((exp(r*dt)-d)/(u-d), isRound, dp)
   f=[[0 for j in range(0,i+1)] for i in range(0,N+1)]
   for j in range(0,N+1):
      f[N][j]=calculate(max(K-S0*u**j*d**(N-j),0), isRound, dp)
   for i in range(N-1, 0-1, -1):
      for j in range(0, i+1):
         f[i][j]=calculate(exp(-r*dt)*(p*f[i+1][j+1]+(1-p)*f[i+1][j]), isRound, dp)
   return f[0][0]

S0,K,r,sigma,T,N=50, 50, 0.1, 0.4, 1, 3
print(CRR_Euro_Put(S0,K,r,sigma,T,N))
print(CRR_Euro_Put(S0,K,r,sigma,T,N,isRound=True))
print(CRR_Euro_Put(S0,K,r,sigma,T,N=12))

## CRR AMERICAN CALL OPTION 

In [ ]:
from math import exp, sqrt
def calculate(x, isRound=False, dp=4):
    return round(x,dp) if isRound else x
def CRR_Amer_Call(S0, K, r, sigma, T, N, isRound=False, dp=4):
   dt=calculate(T/N, isRound, dp)
   u=calculate(exp(sigma*sqrt(dt)), isRound, dp)
   d=calculate(1/u, isRound, dp)
   p=calculate((exp(r*dt)-d)/(u-d), isRound, dp)
   f=[[0 for j in range(0,i+1)] for i in range(0,N+1)]
   for j in range(0,N+1):
      f[N][j]=calculate(max(S0*u**j*d**(N-j)-K,0), isRound, dp)
   for i in range(N-1, 0-1, -1):
      for j in range(0, i+1):
         f[i][j]=calculate(
                max(S0*u**j*d**(i-j)-K, exp(-r*dt)*(p*f[i+1][j+1]+(1-p)*f[i+1][j])), 
                          isRound, dp)
   return f[0][0]

S0,K,r,sigma,T,N=50, 50, 0.1, 0.4, 1, 3
print(CRR_Amer_Call(S0,K,r,sigma,T,N))
print(CRR_Amer_Call(S0,K,r,sigma,T,N,isRound=True))
print(CRR_Amer_Call(S0,K,r,sigma,T,N=12))


## CRR AMERICAN PUT OPTION 

In [ ]:


from math import exp, sqrt
def calculate(x, isRound=False, dp=4):
    return round(x,dp) if isRound else x
def CRR_Amer_Put(S0, K, r, sigma, T, N, isRound=False, dp=4):
   dt=calculate(T/N, isRound, dp)
   u=calculate(exp(sigma*sqrt(dt)), isRound, dp)
   d=calculate(1/u, isRound, dp)
   p=calculate((exp(r*dt)-d)/(u-d), isRound, dp)
   f=[[0 for j in range(0,i+1)] for i in range(0,N+1)]
   for j in range(0,N+1):
      f[N][j]=calculate(max(K-S0*u**j*d**(N-j),0), isRound, dp)
   for i in range(N-1, 0-1, -1):
      for j in range(0, i+1):
         f[i][j]=calculate(
                max(K-S0*u**j*d**(i-j), exp(-r*dt)*(p*f[i+1][j+1]
                          +(1-p)*f[i+1][j])), isRound, dp)
   return f[0][0]

S0,K,r,sigma,T,N=50, 50, 0.1, 0.4, 1, 3
print(CRR_Amer_Put(S0,K,r,sigma,T,N))
print(CRR_Amer_Put(S0,K,r,sigma,T,N,isRound=True))
print(CRR_Amer_Put(S0,K,r,sigma,T,N=12))


## TRINOMIAL TREE EUROPEAN CALL OPTION 

In [ ]:

from math import exp, sqrt
def calculate(x, isRound=False, dp=4):
    return round(x,dp) if isRound else x
def TriTree_Euro_Call(S0, K, r, sigma, T, N, isRound=False, dp=4):
   dt=calculate(T/N, isRound, dp)
   u=calculate(exp(sigma*sqrt(3*dt)), isRound, dp)
   d=calculate(1/u, isRound, dp)
   pu=calculate(1/6+(r-sigma**2/2)*sqrt(dt/(12*sigma**2)), isRound, dp)
   pm=calculate(2/3, isRound, dp)
   pd=calculate(1/6-(r-sigma**2/2)*sqrt(dt/(12*sigma**2)), isRound, dp)
   f=[[0 for j in range(-i,i+1)] for i in range(0,N+1)]
   for j in range(-N,N+1):
      f[N][j+N]=calculate(max(S0*u**j-K,0), isRound, dp)
   for i in range(N-1, 0-1, -1):
      for j in range(-i, i+1):
         f[i][j+i]=calculate(exp(-r*dt)*(pu*f[i+1][j+1+i+1]
                                        +pm*f[i+1][j+i+1]
                                        +pd*f[i+1][j-1+i+1]), isRound, dp)
   return f[0][0]

S0,K,r,sigma,T,N=50, 50, 0.1, 0.4, 1, 3
print(TriTree_Euro_Call(S0,K,r,sigma,T,N))
print(TriTree_Euro_Call(S0,K,r,sigma,T,N,isRound=True))
print(TriTree_Euro_Call(S0,K,r,sigma,T,N=12))


## FINITE DIFFERENCE EXPLICIT EUROPEAN CALL OPTION 

In [ ]:
from math import exp, sqrt, floor
def calculate(x, isRound=False, dp=4):
    return round(x,dp) if isRound else x
def FDExplicit_Euro_Call(S0, K, r, sigma, T, M, N, isRound=False, dp=4):
   dt=calculate(T/N, isRound, dp)
   print('dt:', dt)
   Smax=calculate(2*K, isRound, dp)
   print('Smax:', Smax)
   dS=calculate(Smax/M, isRound, dp)
   print('dS:', dS)
   f=[[0 for j in range(0,M+1)] for i in range(0,N+1)]
   for j in range(0,M+1):
      f[N][j]=calculate(max(j*dS-K,0), isRound, dp)
      print(j, f[N][j])
   for i in range(N-1, 0-1, -1):
      print('i:', i)
      f[i][0]=0
      print(f[i][0])
      f[i][M]=calculate(Smax-K*exp(-r*(N-i)*dt), isRound, dp)
      for j in range(1, M-1+1):
          f[i][j]=calculate(1/2*dt*(sigma**2*j**2-r*j)*f[i+1][j-1]
                           +(1-(sigma**2*j**2+r)*dt)*f[i+1][j]
                           +1/2*dt*(sigma**2*j**2+r*j)*f[i+1][j+1], isRound, dp)
          print(f[i][j])
      print(f[i][M])
   k=floor(S0/dS)
   print('k:', k)
   return calculate(f[0][k]+(f[0][k+1]-f[0][k])/dS*(S0-k*dS), isRound, dp)

S0,K,r,sigma,T,M,N=50, 50, 0.1, 0.4, 1, 4, 3
print(FDExplicit_Euro_Call(S0,K,r,sigma,T,M,N,isRound=True))

In [ ]:
from math import exp, sqrt, floor
import numpy as np
from numpy.linalg import inv
def calculate(x, isRound=False, dp=4):
    return round(x,dp) if isRound else x
def FDImplicit_Euro_Call(S0, K, r, sigma, T, M, N, isRound=False, dp=4):
   dt=calculate(T/N, isRound, dp)
   print('dt:', dt)
   Smax=calculate(2*K, isRound, dp)
   print('Smax:', Smax)
   dS=calculate(Smax/M, isRound, dp)
   print('dS', dS)
   f=np.zeros((M+1,N+1))
   for j in range(0,M+1):
      f[j,N]=calculate(max(j*dS-K,0), isRound, dp)
      print(j, f[j,N])
   A=np.zeros((M+1,M+1))
   A[0,0]=1
   A[M,M]=1
   for j in range(1, M-1+1):
      A[j,j-1]= calculate(1/2*dt*(r*j-sigma**2*j**2), isRound, dp)    #aj
      A[j,j]= calculate(1+dt*(sigma**2*j**2+r), isRound, dp)          #bj
      A[j,j+1]= calculate(-1/2*dt*(r*j+sigma**2*j**2), isRound, dp)   #cj
   print('A:', A)
   Ainv=inv(A).round(4)
   print('Ainv:', Ainv)
   for i in range(N-1, 0-1, -1):
      print('i:', i)
      Fhat=f[:,[i+1]]
      Fhat[0,0]=0
      Fhat[M,0]=Smax-K*exp(-r*(N-i)*dt)
      print('Fhat:', Fhat)
      f[:,[i]]=(Ainv@Fhat).round(4)
      print('f:', f)

   k=floor(S0/dS)
   print('k:', k)
   return calculate(f[k,0]+(f[k+1,0]-f[k,0])/dS*(S0-k*dS), isRound, dp)

S0,K,r,sigma,T,M,N=50, 50, 0.1, 0.4, 1, 4, 3
print(FDImplicit_Euro_Call(S0,K,r,sigma,T,M,N,isRound=True))

In [ ]:
exp_risk = np.arange(0, 1, 0.1)
rf=0.05
rf_slope=(c-2*b*rf+a*rf**2)**(1/2)
rf_return = rf+exp_risk*rf_slope
tangency_rf_return=(c-b*rf)/(b-a*rf)
tangency_rf_risk=((c-2*b*rf+a*rf**2)/(b-a*rf)**2)**(1/2)
plt.scatter(mvp_risk, mvp_return)
plt.plot(exp_risk, rf_return)
plt.scatter(tangency_rf_risk, tangency_rf_return)
plt.plot(risk, exp_returns)
-------------------------------